# 01 — Première exploration des données DVF

Données : *Demandes de valeurs foncières géolocalisées* (data.gouv.fr), ventes immobilières
de la région PACA (départements 04, 05, 06, 13, 83, 84), années 2021 à 2025.

Les fichiers sont téléchargés avec `python src/download_data.py`.

## Chargement

In [1]:
import pandas as pd
from pathlib import Path

# Remonter les dossiers jusqu'à trouver data/raw (fonctionne où que soit le notebook)
racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

fichiers = sorted((racine / "data" / "raw").glob("*.csv.gz"))
print(f"{len(fichiers)} fichiers trouvés dans {racine / 'data' / 'raw'}")

# Les codes (postal, commune, département, parcelle) sont des identifiants, pas des nombres :
# on les lit comme du texte pour garder les zéros de tête ("04" et non 4)
types_texte = {"code_postal": str, "code_commune": str, "code_departement": str, "id_parcelle": str}
df = pd.concat(
    (pd.read_csv(f, low_memory=False, dtype=types_texte) for f in fichiers),
    ignore_index=True,
)
print(f"Lignes brutes : {len(df):,}")
df.shape

30 fichiers trouvés dans c:\Users\salwa\estimateur-immo-dvf\data\raw
Lignes brutes : 1,720,509


(1720509, 40)

## Aperçu et colonnes

In [2]:
df.head()

,id_mutation,date_mutation,numero_disposition,nature_mutation,valeur_fonciere,adresse_numero,adresse_suffixe,adresse_nom_voie,adresse_code_voie,code_postal,...,type_local,surface_reelle_bati,nombre_pieces_principales,code_nature_culture,nature_culture,code_nature_culture_speciale,nature_culture_speciale,surface_terrain,longitude,latitude
0,2021-40424,2021-01-05,1,Vente,81960.0,7037.0,NaN,LES PLAINES,B116,04800,...,Appartement,28.0,1.0,NaN,NaN,NaN,NaN,NaN,5.894395,43.759373
1,2021-40425,2021-01-05,1,Vente,105152.0,5282.0,NaN,LE VILLAGE,B155,04340,...,Appartement,76.0,3.0,S,sols,NaN,NaN,155.0,6.506323,44.398306
2,2021-40425,2021-01-05,1,Vente,105152.0,5282.0,NaN,LE VILLAGE,B155,04340,...,Appartement,35.0,2.0,S,sols,NaN,NaN,155.0,6.506323,44.398306
3,2021-40426,2021-01-05,1,Vente,405500.0,5.0,NaN,LES MOIS,B143,04400,...,Maison,86.0,3.0,S,sols,NaN,NaN,401.0,6.610453,44.359152
4,2021-40427,2021-01-06,1,Vente,4500.0,NaN,NaN,L ESCALLE,B045,04190,...,NaN,NaN,NaN,T,terres,NaN,NaN,4069.0,5.924195,43.979626


In [3]:
df.columns.tolist()

['id_mutation',
 'date_mutation',
 'numero_disposition',
 'nature_mutation',
 'valeur_fonciere',
 'adresse_numero',
 'adresse_suffixe',
 'adresse_nom_voie',
 'adresse_code_voie',
 'code_postal',
 'code_commune',
 'nom_commune',
 'code_departement',
 'ancien_code_commune',
 'ancien_nom_commune',
 'id_parcelle',
 'ancien_id_parcelle',
 'numero_volume',
 'lot1_numero',
 'lot1_surface_carrez',
 'lot2_numero',
 'lot2_surface_carrez',
 'lot3_numero',
 'lot3_surface_carrez',
 'lot4_numero',
 'lot4_surface_carrez',
 'lot5_numero',
 'lot5_surface_carrez',
 'nombre_lots',
 'code_type_local',
 'type_local',
 'surface_reelle_bati',
 'nombre_pieces_principales',
 'code_nature_culture',
 'nature_culture',
 'code_nature_culture_speciale',
 'nature_culture_speciale',
 'surface_terrain',
 'longitude',
 'latitude']

> ✍️ **À toi** — décris en quelques lignes les grandes familles de colonnes (la vente, l'adresse, le bien, le terrain, les lots, le GPS) et indique quelle colonne sera la variable à prédire.
>
> *(remplace ce bloc par ton commentaire)*

## Types de ventes et de biens

In [4]:
df["nature_mutation"].value_counts()

nature_mutation
Vente                                 1576006
Vente en l'état futur d'achèvement     124490
Echange                                 10846
Adjudication                             5979
Vente terrain à bâtir                    2514
Expropriation                             674
Name: count, dtype: int64

In [5]:
df["type_local"].value_counts(dropna=False)

type_local
Dépendance                                  608861
NaN                                         438855
Appartement                                 395718
Maison                                      218406
Local industriel. commercial ou assimilé     58669
Name: count, dtype: int64

## Une vente = plusieurs lignes ?

In [6]:
lignes_par_vente = df.groupby("id_mutation").size()
print(f"Nombre de ventes uniques : {lignes_par_vente.shape[0]:,}")
print(f"Nombre de lignes         : {len(df):,}")
lignes_par_vente.value_counts().head(10)

Nombre de ventes uniques : 680,843
Nombre de lignes         : 1,720,509


1     251999
2     217577
3     108913
4      49996
5      15410
6      12365
7       5448
8       4698
9       3191
10      2189
Name: count, dtype: int64

In [7]:
# Exemple concret : une vente qui occupe plusieurs lignes, avec le même prix répété
exemple = lignes_par_vente[lignes_par_vente.between(2, 4)].index[0]
df.loc[df["id_mutation"] == exemple,
       ["id_mutation", "date_mutation", "valeur_fonciere", "type_local", "surface_reelle_bati", "nature_culture"]]

,id_mutation,date_mutation,valeur_fonciere,type_local,surface_reelle_bati,nature_culture
246386,2021-1397162,2021-01-05,365000.0,Maison,135.0,sols
246387,2021-1397162,2021-01-05,365000.0,Maison,135.0,terrains d'agrément
246388,2021-1397162,2021-01-05,365000.0,NaN,NaN,terres


> ✍️ **À toi** — explique avec tes mots pourquoi une même vente apparaît sur plusieurs lignes, et pourquoi c'est un piège si on calcule des prix moyens ou si on entraîne un modèle sur ces lignes telles quelles.
>
> *(remplace ce bloc par ton commentaire)*